In [1]:
!pip install azure-eventhub


In [2]:
import time
import json
import requests
from datetime import datetime
from azure.eventhub import EventHubProducerClient, EventData

In [3]:
CONNECTION_STR = "Endpoint=sb://esehamb2unlhl9w9scaxw5.servicebus.windows.net/;SharedAccessKeyName=key_817bf219-e1e6-4391-a144-87ac35945e47;SharedAccessKey=R1wwvqyLo1Uc23H77dHEVEGc24BF0q6qL+AEhNaunHw=;EntityPath=esehamb2unlhl9w9scaxw5_eh"
EVENT_HUB_NAME = "esehamb2unlhl9w9scaxw5_eh"


In [8]:
def get_carbon_intensity(battery_level):
    url = "https://api.carbonintensity.org.uk/intensity"
    headers = {'Accept': 'application/json'}
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            data = response.json()['data'][0]
            intensity = data['intensity']['actual']

            charge_now, reason = should_charge(intensity, battery_level)
            status = "CHARGING" if charge_now else "IDLE"

            payload = {
                "timestamp": data['from'],
                "actual_intensity": intensity,
                "forecast_intensity": data['intensity']['forecast'],
                "index_rating": data['intensity']['index'],
                "device_id": "EV_CHARGER_HOME_01",
                "battery_level": battery_level,
                "charge_now": charge_now,
                "reason": reason,
                "simulated_charging_status": status
            }
            return payload
    except Exception as e:
        print(f"Error fetching API: {e}")
        return None

In [17]:
def send_to_fabric_eventstream():
    client = EventHubProducerClient.from_connection_string(
        conn_str = CONNECTION_STR,
        eventhub_name = EVENT_HUB_NAME
    )

    print(" Starting live streaming to Fabric Eventstream...")

    battery_level = None
    charge_now = False   # start assuming idle

    with client:
        while True:
            battery_level = simulate_battery_state(battery_level, charge_now)
            payload = get_carbon_intensity(battery_level)
            if payload:
                charge_now = payload["charge_now"]   # carry forward for next iteration
                event_data = EventData(json.dumps(payload))

                event_batch = client.create_batch()
                event_batch.add(event_data)
                client.send_batch(event_batch)

                print(f"[{payload['timestamp']}] {payload['actual_intensity']} gCO2/kWh ({payload['index_rating']}) | Battery: {payload['battery_level']}% | {payload['simulated_charging_status']}")

            time.sleep(60)

# --- RUN THE STREAMER ---
send_to_fabric_eventstream()

 Starting live streaming to Fabric Eventstream...
[2026-08-24T23:30Z] 73 gCO2/kWh (low) | Battery: 32% | IDLE
[2026-08-24T23:30Z] 73 gCO2/kWh (low) | Battery: 32% | IDLE
[2026-08-24T23:30Z] 73 gCO2/kWh (low) | Battery: 32% | IDLE
[2026-08-24T23:30Z] 73 gCO2/kWh (low) | Battery: 31% | IDLE
[2026-08-24T23:30Z] 73 gCO2/kWh (low) | Battery: 30% | IDLE
[2026-08-24T23:30Z] 73 gCO2/kWh (low) | Battery: 29% | IDLE
[2026-08-24T23:30Z] 73 gCO2/kWh (low) | Battery: 28% | IDLE


KeyboardInterrupt: 

In [16]:
import random

def simulate_battery_state(prev_level, charge_now):
    if prev_level is None:
        prev_level = random.randint(20, 60)

    if charge_now:
        delta = random.choice([2, 3, 3, 4])   # charging: battery rises
    else:
        delta = random.choice([-1, -1, 0])    # idle: slow drain or flat

    new_level = max(0, min(100, prev_level + delta))
    return new_level

recent_intensities = []

def should_charge(intensity, battery_level, battery_cap=80):
    if battery_level >= battery_cap:
        return False, "Battery sufficiently charged"

    recent_intensities.append(intensity)
    if len(recent_intensities) > 20:
        recent_intensities.pop(0)

    avg_recent = sum(recent_intensities) / len(recent_intensities)

    if intensity < avg_recent:
        return True, f"Below recent average ({avg_recent:.0f} gCO2/kWh)"
    return False, f"Above recent average ({avg_recent:.0f} gCO2/kWh)"